# Model 2 - VGG Sequential
Notebook train và test VGG Sequential.

In [1]:
# 1. Import và cấu hình
import os
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report

from metrics import calculate_metrics

DATA_DIR = '../../../data/VN_trash_classification_preprocessing'
CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCHS = 30
PATIENCE = 7
BATCH_SIZE = 32
LR = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# 2. Xử lý ảnh và load dữ liệu
normalize = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])

train_ds = ImageFolder(DATA_DIR + '/train', transform=train_transform)
val_ds = ImageFolder(DATA_DIR + '/val', transform=test_transform)
test_ds = ImageFolder(DATA_DIR + '/test', transform=test_transform)

# Nạp ảnh bằng nhiều tiến trình để GPU không phải chờ CPU
loader_options = {'num_workers': 8, 'pin_memory': True, 'persistent_workers': True}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, **loader_options)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, **loader_options)

class_names = train_ds.classes
print('Device:', device, '| Classes:', class_names)
print(len(train_ds), len(val_ds), len(test_ds))

Device: cuda | Classes: ['Alu', 'Carton', 'Foam_box', 'Milk_box', 'Other', 'PET', 'Paper', 'Paper_cup', 'Plastic_cup']
9542 1685 864


In [3]:
# 3. Tạo VGG Sequential
class VGGSequential(nn.Module):
    def __init__(self, number_of_classes):
        super().__init__()

        # 5 stage VGG16: số conv mỗi stage là 2/2/3/3/3, sau mỗi stage có MaxPool
        layers = []
        in_channels = 3
        for width, count in ((64, 2), (128, 2), (256, 3), (512, 3), (512, 3)):
            for _ in range(count):
                layers += [
                    nn.Conv2d(in_channels, width, kernel_size=3, padding=1),
                    nn.BatchNorm2d(width),
                    nn.ReLU(inplace=True),
                ]
                in_channels = width
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.6),
            nn.Linear(128, number_of_classes),
        )

    def forward(self, images):
        return self.classifier(self.features(images))


model = VGGSequential(len(class_names)).to(device)
print(model)

VGGSequential(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2

In [4]:
# 4. Loss, optimizer và scheduler
counts = Counter(train_ds.targets)
weights = torch.tensor(
    [len(train_ds) / (len(class_names) * counts[i]) for i in range(len(class_names))],
    dtype=torch.float32,
    device=device,
)

loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

In [5]:
# 5. Train và validation
def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    return total_loss / seen, correct / seen


best_acc = -1.0
best_state = None
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1}/{EPOCHS} | '
        f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | '
        f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}'
    )

    # Dừng khi validation accuracy không tăng trong PATIENCE epoch liên tiếp.
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping tại epoch {epoch + 1}')
        break

model.load_state_dict(best_state)
print('Best epoch:', best_epoch, '| Best validation accuracy:', f'{best_acc:.2%}')

Epoch 1/30 | Train Loss: 1.9486 | Train Acc: 29.57% | Val Loss: 1.9022 | Val Acc: 31.57%


Epoch 2/30 | Train Loss: 1.8327 | Train Acc: 36.05% | Val Loss: 1.7275 | Val Acc: 40.47%


Epoch 3/30 | Train Loss: 1.7341 | Train Acc: 39.59% | Val Loss: 1.9202 | Val Acc: 38.81%


Epoch 4/30 | Train Loss: 1.6661 | Train Acc: 42.68% | Val Loss: 1.8546 | Val Acc: 38.58%


Epoch 5/30 | Train Loss: 1.5966 | Train Acc: 45.07% | Val Loss: 1.4581 | Val Acc: 50.15%


...

Epoch 26/30 | Train Loss: 0.5245 | Train Acc: 83.87% | Val Loss: 0.6498 | Val Acc: 79.53%


Epoch 27/30 | Train Loss: 0.5053 | Train Acc: 84.02% | Val Loss: 0.6517 | Val Acc: 79.47%


Epoch 28/30 | Train Loss: 0.4922 | Train Acc: 84.98% | Val Loss: 0.6554 | Val Acc: 79.70%


Epoch 29/30 | Train Loss: 0.4855 | Train Acc: 85.53% | Val Loss: 0.6686 | Val Acc: 79.47%


Epoch 30/30 | Train Loss: 0.4763 | Train Acc: 85.40% | Val Loss: 0.6561 | Val Acc: 79.47%
Best epoch: 25 | Best validation accuracy: 79.70%


In [6]:
# 6. Test và đánh giá
model.eval()
true_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predictions.cpu().tolist())

metrics = calculate_metrics(true_labels, predicted_labels)

print()
print('Test Result')
print(f'Test Accuracy: {metrics["accuracy"]:.2%}')
print(f'Macro Precision: {metrics["macro_precision"]:.4f}')
print(f'Macro Recall: {metrics["macro_recall"]:.4f}')
print(f'Macro F1: {metrics["macro_f1"]:.4f}')
print(f'Weighted F1: {metrics["weighted_f1"]:.4f}')
print()
print(classification_report(true_labels, predicted_labels, target_names=class_names, digits=4, zero_division=0))


Test Result
Test Accuracy: 72.34%
Macro Precision: 0.7202
Macro Recall: 0.7175
Macro F1: 0.7109
Weighted F1: 0.7242

              precision    recall  f1-score   support

         Alu     0.7843    0.8081    0.7960        99
      Carton     0.8866    0.7963    0.8390       108
    Foam_box     0.7358    0.4483    0.5571        87
    Milk_box     0.8866    0.8515    0.8687       101
       Other     0.4430    0.7000    0.5426        50
         PET     0.5739    0.6471    0.6083       102
       Paper     0.7500    0.7075    0.7282       106
   Paper_cup     0.7931    0.8762    0.8326       105
 Plastic_cup     0.6286    0.6226    0.6256       106

    accuracy                         0.7234       864
   macro avg     0.7202    0.7175    0.7109       864
weighted avg     0.7373    0.7234    0.7242       864



In [7]:
# 7. Lưu model
torch.save(model.state_dict(), CHECKPOINT_DIR + '/vgg_sequential.pt')